In [1]:
from pathlib import Path
import pandas as pd

project_root = Path(r"C:\Users\HP-ZBOOK i7\graphrag_test")

pilot1_duplicates = pd.read_csv(
    project_root
    / "pilot_01"
    / "inspection"
    / "10_possible_duplicates.csv"
)

pilot2_duplicates = pd.read_csv(
    project_root
    / "pilot_02"
    / "inspection"
    / "10_possible_duplicates.csv"
)

pilot2_entities = pd.read_csv(
    project_root
    / "pilot_02"
    / "inspection"
    / "01_entities_all.csv"
)

print(
    "Pilot 1 automatically detected duplicate candidates:",
    len(pilot1_duplicates),
)

print(
    "Pilot 2 automatically detected duplicate candidates:",
    len(pilot2_duplicates),
)

print("\nPilot 2 duplicate-file columns:")
print(pilot2_duplicates.columns.tolist())

pilot2_duplicates

Pilot 1 automatically detected duplicate candidates: 2
Pilot 2 automatically detected duplicate candidates: 2

Pilot 2 duplicate-file columns:
['title', 'normalized_title', 'type', 'description', 'frequency', 'degree']


,title,normalized_title,type,description,frequency,degree
0,WOHNUNGSEIGENTUMSGESETZ (NOVELLE 2022),WOHNUNGSEIGENTUMSGESETZ NOVELLE 2022,POLICY,Policy type: law; Jurisdiction: Austria; Statu...,1,1
1,WOHNUNGSEIGENTUMSGESETZ-NOVELLE 2022,WOHNUNGSEIGENTUMSGESETZ NOVELLE 2022,POLICY,Policy type: law (amendment); Jurisdiction: Au...,1,2


In [2]:
alias_patterns = {
    "Photovoltaic concepts": r"PHOTOVOLTAIK|FOTOVOLTAIK|\bPV\b",
    "Austrian PV Strategy": r"PHOTOVOLTAIK.?STRATEGIE|PV.?STRATEGIE",
    "BMK and ministry": r"\bBMK\b|BUNDESMINISTERIUM",
    "Energy communities": r"ENERGIEGEMEINSCHAFT",
    "Network infrastructure plan": r"NETZINFRASTRUKTURPLAN|\bNIP\b|\bÖNIP\b",
    "EAG": r"ERNEUERBAREN.?AUSBAU.?GESETZ|\bEAG\b",
    "Federal states": r"BUNDESLAND|BUNDESLÄNDER",
    "Electricity Industry Act": r"ELEKTRIZITÄTSWIRTSCHAFTSGESETZ|\bELWG\b|\bELWOG\b",
}

alias_results = []

for family, pattern in alias_patterns.items():
    matches = pilot2_entities[
        pilot2_entities["title"].str.contains(
            pattern,
            case=False,
            regex=True,
            na=False,
        )
    ].copy()

    matches["alias_family"] = family
    alias_results.append(matches)

alias_candidates = (
    pd.concat(alias_results, ignore_index=True)
    .drop_duplicates(subset="id")
    .sort_values(
        by=["alias_family", "degree", "frequency"],
        ascending=[True, False, False],
    )
)

print("Alias candidates found:", len(alias_candidates))

alias_candidates[
    [
        "alias_family",
        "title",
        "type",
        "frequency",
        "degree",
        "description",
    ]
]

Alias candidates found: 164


,alias_family,title,type,frequency,degree,description
141,BMK and ministry,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ORGANIZATION,2,6,"The BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT,..."
144,BMK and ministry,BMK-AUSBILDUNGSINITIATIVE „JUST TRANSITION“,SUPPORT_SCHEME,1,6,Training initiative by Austria’s BMK focusing ...
142,BMK and ministry,BMK,ORGANIZATION,3,3,BMK (Austrian Federal Ministry for Climate Act...
143,BMK and ministry,STAKEHOLDERRUNDEN DES BMK,SUPPORT_SCHEME,1,2,Stakeholder rounds organized by the BMK serve ...
145,BMK and ministry,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ORGANIZATION,1,1,The Austrian Federal Ministry for Climate Acti...
...,...,...,...,...,...,...
36,Photovoltaic concepts,INVESTITIONSBARRIEREFREIER BETRIEB VON PV-ANLA...,TARGET,1,0,Target: elimination of investment barriers for...
39,Photovoltaic concepts,HOHE AKZEPTANZ DER PHOTOVOLTAIK IN DER BEVÖLKE...,MARKET_METRIC,1,0,Metric: social acceptance of photovoltaic depl...
85,Photovoltaic concepts,PV-AUSTRIA,ORGANIZATION,1,0,PV Austria is referenced as participating in a...
119,Photovoltaic concepts,PV-BRANCHENVERTRETUNG,STAKEHOLDER,1,0,PV-sector advocacy or representative bodies in...


In [3]:
canonical_groups = {
    "PHOTOVOLTAIK": [
        "PHOTOVOLTAIK",
        "PHOTOVOLTAIK (PV)",
    ],

    "PV-ANLAGE": [
        "PV-ANLAGE",
        "PV-ANLAGEN",
        "PHOTOVOLTAIKANLAGE",
        "PHOTOVOLTAIKANLAGEN",
        "FOTOVOLTAIKANLAGE",
    ],

    "ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE": [
        "ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE",
        "PHOTOVOLTAIK-STRATEGIE",
        "PV-STRATEGIE",
    ],

    "ENERGIEGEMEINSCHAFTEN": [
        "ENERGIEGEMEINSCHAFTEN",
        "ENERGIEGEMEINSCHAFTEN (EGS)",
    ],

    "BUNDESLAND": [
        "BUNDESLAND",
        "BUNDESLÄNDER",
    ],

    "WOHNUNGSEIGENTUMSGESETZ-NOVELLE 2022": [
        "WOHNUNGSEIGENTUMSGESETZ (NOVELLE 2022)",
        "WOHNUNGSEIGENTUMSGESETZ-NOVELLE 2022",
    ],
}

duplicate_review_rows = []

for canonical_title, variants in canonical_groups.items():
    matches = pilot2_entities[
        pilot2_entities["title"].isin(variants)
    ].copy()

    if len(matches) == 0:
        continue

    matches["canonical_title"] = canonical_title

    matches["resolution_decision"] = matches["title"].apply(
        lambda title:
        "KEEP_CANONICAL"
        if title == canonical_title
        else "MERGE_AS_ALIAS"
    )

    duplicate_review_rows.append(matches)

duplicate_review = (
    pd.concat(
        duplicate_review_rows,
        ignore_index=True,
    )
    .sort_values(
        by=["canonical_title", "degree"],
        ascending=[True, False],
    )
)

duplicate_review[
    [
        "canonical_title",
        "title",
        "type",
        "frequency",
        "degree",
        "resolution_decision",
    ]
]

,canonical_title,title,type,frequency,degree,resolution_decision
10,BUNDESLAND,BUNDESLÄNDER,GEOGRAPHIC_AREA,2,5,MERGE_AS_ALIAS
11,BUNDESLAND,BUNDESLAND,GEOGRAPHIC_AREA,1,2,KEEP_CANONICAL
8,ENERGIEGEMEINSCHAFTEN,ENERGIEGEMEINSCHAFTEN,STAKEHOLDER,3,11,KEEP_CANONICAL
9,ENERGIEGEMEINSCHAFTEN,ENERGIEGEMEINSCHAFTEN (EGS),STAKEHOLDER,1,6,MERGE_AS_ALIAS
0,PHOTOVOLTAIK,PHOTOVOLTAIK,TECHNOLOGY,11,85,KEEP_CANONICAL
1,PHOTOVOLTAIK,PHOTOVOLTAIK (PV),TECHNOLOGY,2,18,MERGE_AS_ALIAS
3,PV-ANLAGE,PV-ANLAGE,TECHNOLOGY,5,26,KEEP_CANONICAL
5,PV-ANLAGE,FOTOVOLTAIKANLAGE,TECHNOLOGY,1,16,MERGE_AS_ALIAS
4,PV-ANLAGE,PHOTOVOLTAIKANLAGEN,TECHNOLOGY,2,8,MERGE_AS_ALIAS
2,PV-ANLAGE,PV-ANLAGEN,TECHNOLOGY,3,5,MERGE_AS_ALIAS


In [4]:
careful_review = pilot2_entities[
    pilot2_entities["title"].str.contains(
        r"(^BMK$|BUNDESMINISTERIUM|NETZINFRASTRUKTURPLAN|\bNIP\b|\bÖNIP\b)",
        case=False,
        regex=True,
        na=False,
    )
].copy()

careful_review[
    [
        "title",
        "type",
        "frequency",
        "degree",
        "description",
    ]
].sort_values(
    by=["degree", "frequency"],
    ascending=False,
)

C:\Users\HP-ZBOOK i7\AppData\Local\Temp\ipykernel_56624\528637506.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  pilot2_entities["title"].str.contains(


,title,type,frequency,degree,description
65,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTU...,POLICY,3,8,The INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTR...
1,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ORGANIZATION,2,6,"The BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT,..."
33,BMK,ORGANIZATION,3,3,BMK (Austrian Federal Ministry for Climate Act...
160,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTU...,POLICY,1,3,Policy type: plan; Jurisdiction: Austria; Stat...
310,NIP,POLICY,1,3,Policy type: plan; Jurisdiction: Austria; Stat...
507,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTU...,POLICY,1,2,Policy type: plan; Jurisdiction: Austria; Stat...
11,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTU...,POLICY,1,1,Policy type: plan; Jurisdiction: Austria; Stat...
270,REGELMÄSSIGE ANPASSUNG DER FLÄCHENPOTENTIALE I...,POLICY,1,1,Policy type: planning requirement; Jurisdictio...
537,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENE...",ORGANIZATION,1,1,The Austrian Federal Ministry for Climate Acti...


In [5]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 300)

print(
    careful_review[
        [
            "title",
            "type",
            "frequency",
            "degree",
        ]
    ].to_string(index=True)
)

                                                                                               title          type  frequency  degree
1    BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE (BMK)  ORGANIZATION          2       6
11                                        INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (ÖNIP)        POLICY          1       1
33                                                                                               BMK  ORGANIZATION          3       3
65                                         INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (NIP)        POLICY          3       8
160                                   INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (NIP/ÖNIP)        POLICY          1       3
270                                              REGELMÄSSIGE ANPASSUNG DER FLÄCHENPOTENTIALE IM NIP        POLICY          1       1
310                                                           

In [6]:
confirmed_aliases = duplicate_review[
    duplicate_review["resolution_decision"]
    == "MERGE_AS_ALIAS"
].copy()

print(
    "Confirmed alias records from clear groups:",
    len(confirmed_aliases),
)

print(
    "Canonical groups represented:",
    confirmed_aliases["canonical_title"].nunique(),
)

print("\nConfirmed aliases by canonical entity:")
print(
    confirmed_aliases[
        "canonical_title"
    ].value_counts()
)

Confirmed alias records from clear groups: 8
Canonical groups represented: 6

Confirmed aliases by canonical entity:
canonical_title
PV-ANLAGE                                 3
BUNDESLAND                                1
ENERGIEGEMEINSCHAFTEN                     1
PHOTOVOLTAIK                              1
WOHNUNGSEIGENTUMSGESETZ-NOVELLE 2022      1
ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE    1
Name: count, dtype: int64


In [7]:
additional_alias_mappings = [
    {
        "canonical_title": (
            "BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, "
            "ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE (BMK)"
        ),
        "title": "BMK",
        "resolution_decision": "MERGE_AS_ALIAS",
        "resolution_notes": "BMK is the abbreviation of the complete ministry name.",
    },
    {
        "canonical_title": (
            "BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, "
            "ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE (BMK)"
        ),
        "title": (
            "BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, "
            "ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE"
        ),
        "resolution_decision": "MERGE_AS_ALIAS",
        "resolution_notes": "Same ministry name without the BMK abbreviation.",
    },
    {
        "canonical_title": (
            "INTEGRIERTER ÖSTERREICHISCHER "
            "NETZINFRASTRUKTURPLAN (ÖNIP)"
        ),
        "title": (
            "INTEGRIERTER ÖSTERREICHISCHER "
            "NETZINFRASTRUKTURPLAN (NIP)"
        ),
        "resolution_decision": "MERGE_AS_ALIAS",
        "resolution_notes": "NIP and ÖNIP refer to the same Austrian network infrastructure plan.",
    },
    {
        "canonical_title": (
            "INTEGRIERTER ÖSTERREICHISCHER "
            "NETZINFRASTRUKTURPLAN (ÖNIP)"
        ),
        "title": (
            "INTEGRIERTER ÖSTERREICHISCHER "
            "NETZINFRASTRUKTURPLAN (NIP/ÖNIP)"
        ),
        "resolution_decision": "MERGE_AS_ALIAS",
        "resolution_notes": "Combined abbreviation variant of the same plan.",
    },
    {
        "canonical_title": (
            "INTEGRIERTER ÖSTERREICHISCHER "
            "NETZINFRASTRUKTURPLAN (ÖNIP)"
        ),
        "title": "NIP",
        "resolution_decision": "MERGE_AS_ALIAS",
        "resolution_notes": "Short abbreviation of the same plan.",
    },
    {
        "canonical_title": (
            "INTEGRIERTER ÖSTERREICHISCHER "
            "NETZINFRASTRUKTURPLAN (ÖNIP)"
        ),
        "title": (
            "INTEGRIERTER ÖSTERREICHISCHER "
            "NETZINFRASTRUKTURPLAN"
        ),
        "resolution_decision": "MERGE_AS_ALIAS",
        "resolution_notes": "Same plan name without an abbreviation.",
    },
]

additional_aliases = pd.DataFrame(
    additional_alias_mappings
)

# Add notes to the earlier confirmed mappings
confirmed_aliases = confirmed_aliases.copy()
confirmed_aliases["resolution_notes"] = (
    "Confirmed spelling, abbreviation, or singular/plural alias."
)

confirmed_aliases_for_export = confirmed_aliases[
    [
        "canonical_title",
        "title",
        "resolution_decision",
        "resolution_notes",
    ]
]

all_confirmed_aliases = (
    pd.concat(
        [
            confirmed_aliases_for_export,
            additional_aliases,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=["canonical_title", "title"]
    )
    .sort_values(
        by=["canonical_title", "title"]
    )
    .reset_index(drop=True)
)

print(
    "Total confirmed alias records:",
    len(all_confirmed_aliases),
)

print(
    "Canonical groups:",
    all_confirmed_aliases[
        "canonical_title"
    ].nunique(),
)

print(
    "Minimum proportion of Pilot 2 entities identified as aliases:",
    round(
        len(all_confirmed_aliases)
        / len(pilot2_entities)
        * 100,
        1,
    ),
    "%",
)

all_confirmed_aliases

Total confirmed alias records: 14
Canonical groups: 8
Minimum proportion of Pilot 2 entities identified as aliases: 2.6 %


,canonical_title,title,resolution_decision,resolution_notes
0,BUNDESLAND,BUNDESLÄNDER,MERGE_AS_ALIAS,"Confirmed spelling, abbreviation, or singular/plural alias."
1,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE (BMK)",BMK,MERGE_AS_ALIAS,BMK is the abbreviation of the complete ministry name.
2,"BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE (BMK)","BUNDESMINISTERIUM FÜR KLIMASCHUTZ, UMWELT, ENERGIE, MOBILITÄT, INNOVATION UND TECHNOLOGIE",MERGE_AS_ALIAS,Same ministry name without the BMK abbreviation.
3,ENERGIEGEMEINSCHAFTEN,ENERGIEGEMEINSCHAFTEN (EGS),MERGE_AS_ALIAS,"Confirmed spelling, abbreviation, or singular/plural alias."
4,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (ÖNIP),INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN,MERGE_AS_ALIAS,Same plan name without an abbreviation.
5,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (ÖNIP),INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (NIP),MERGE_AS_ALIAS,NIP and ÖNIP refer to the same Austrian network infrastructure plan.
6,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (ÖNIP),INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (NIP/ÖNIP),MERGE_AS_ALIAS,Combined abbreviation variant of the same plan.
7,INTEGRIERTER ÖSTERREICHISCHER NETZINFRASTRUKTURPLAN (ÖNIP),NIP,MERGE_AS_ALIAS,Short abbreviation of the same plan.
8,PHOTOVOLTAIK,PHOTOVOLTAIK (PV),MERGE_AS_ALIAS,"Confirmed spelling, abbreviation, or singular/plural alias."
9,PV-ANLAGE,FOTOVOLTAIKANLAGE,MERGE_AS_ALIAS,"Confirmed spelling, abbreviation, or singular/plural alias."


In [16]:
alias_output_path = (
    project_root
    / "pilot_02"
    / "inspection"
    / "10_confirmed_alias_resolution.csv"
)

all_confirmed_aliases.to_csv(
    alias_output_path,
    index=False,
    encoding="utf-8-sig",
)

print("Saved confirmed alias mappings to:")
print(alias_output_path)

Saved confirmed alias mappings to:
C:\Users\HP-ZBOOK i7\graphrag_test\pilot_02\inspection\10_confirmed_alias_resolution.csv


# Pilot 2 — Duplicate and Entity-Resolution Audit

## Objective

This inspection evaluated whether Pilot 2 created multiple graph entities for the same underlying real-world concept.

The automatically generated `10_possible_duplicates.csv` file detected only entities whose titles became identical after simple text normalization. A second domain-guided inspection was therefore performed for abbreviations, spelling variants, shortened names and singular/plural forms.

## Automatic duplicate detection

Both Pilot 1 and Pilot 2 contained two automatically detected candidate rows.

In Pilot 2 these were:

- WOHNUNGSEIGENTUMSGESETZ (NOVELLE 2022)
- WOHNUNGSEIGENTUMSGESETZ-NOVELLE 2022

These represent one policy concept expressed using slightly different punctuation and word order.

The automatic detector therefore found only one clear duplicate group.

## Domain-guided alias inspection

The broad pattern search initially returned 164 candidates. These were not 164 duplicates. Most were legitimately different concepts that happened to contain terms such as PV, Photovoltaik, BMK or NIP.

After controlled manual review, 14 alias records were confirmed across eight canonical groups.

Confirmed examples included:

- PHOTOVOLTAIK (PV) → PHOTOVOLTAIK
- PV-ANLAGEN → PV-ANLAGE
- PHOTOVOLTAIKANLAGE → PV-ANLAGE
- PHOTOVOLTAIKANLAGEN → PV-ANLAGE
- PHOTOVOLTAIK-STRATEGIE → ÖSTERREICHISCHE PHOTOVOLTAIK-STRATEGIE
- ENERGIEGEMEINSCHAFTEN (EGS) → ENERGIEGEMEINSCHAFTEN
- BUNDESLÄNDER → BUNDESLAND
- BMK → complete ministry name
- ministry name without BMK → complete ministry name with BMK
- NIP, ÖNIP, NIP/ÖNIP and full-name variants → one network infrastructure plan
- punctuation variants of the 2022 Condominium Act amendment → one policy entity

The 14 confirmed alias records represent approximately 2.6% of the 539 Pilot 2 entities.

This percentage is a minimum confirmed rate, not an estimate of all duplicates in the graph, because only selected high-priority alias families were reviewed.

## Important distinction

Related concepts must not automatically be merged.

For example:

- PV technology and a PV installation are related but conceptually different;
- PV expansion, PV generation and PV market share are different concepts;
- the NIP itself is different from a policy action such as regularly updating area potentials within the NIP;
- ELWG and ElWOG may represent different legal instruments or versions and require legal-context verification before merging;
- BMK initiatives are not aliases of the ministry itself.

Therefore, substring similarity alone is not a reliable entity-resolution method.

## Comparison with Pilot 1

Pilot 1 already showed important unresolved aliases, including:

- BMK versus the full ministry name;
- NIP/ÖNIP variants;
- PV and Photovoltaik variants;
- strategy-name variants.

Pilot 2 improved entity typing but did not solve record linkage. In fact, its larger and more detailed extraction output created more opportunities for spelling, abbreviation and granularity variants.

This demonstrates that improving the extraction schema and prompt does not replace a separate entity-resolution stage.

## Recommended record-linkage process

The controlled knowledge graph should use a canonicalization table containing:

- canonical identifier;
- canonical label;
- alternative labels;
- abbreviations;
- spelling variants;
- source-language variants;
- entity type;
- merge decision;
- merge justification.

Extracted entities should be mapped to canonical identifiers before import into the final controlled KG.

## Conclusion

Pilot 2 successfully creates more domain-relevant entities, but GraphRAG still does not reliably consolidate aliases.

The automatic duplicate detector substantially underestimates the problem: it identified only one obvious duplicate group, while domain-guided inspection confirmed 14 alias records across eight groups.

Entity resolution must therefore remain a distinct downstream stage between GraphRAG extraction and controlled-KG construction.